In [6]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth.csv") #-data.csv")

In [7]:
df_ground_truth.head()

,question,document
0,Is it too late to enroll since I just found ou...,74eb249bbf
1,Can I catch up and start participating even th...,74eb249bbf
2,"If I join now, will I still be eligible to ear...",74eb249bbf
3,Are late sign-ups allowed for students wanting...,74eb249bbf
4,What is the deal with submitting my final proj...,74eb249bbf


In [8]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [9]:
ground_truth[10]

{'question': 'Where can I find the direct Zoom link to join the live workshop sessions?',
 'document': '489dd1c9d9'}

In [50]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)


In [11]:
boost = {'question': 3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

[{'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson notebooks/code.\n3. Read the homework instructions on GitHub.\n4. Submit answers through the course platform before the deadline.\n\nHomework is similar to the lesson flow, but uses a different dataset or slightly different task.'},
 {'id': 'db78580409

In [12]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [13]:
q = ground_truth[0]
q

{'question': 'Is it too late to enroll since I just found out about the program?',
 'document': '74eb249bbf'}

In [14]:
doc_id = q['document']
doc_id

'74eb249bbf'

In [15]:
results = text_search(q['question'])
results

[{'id': '0fab61eca2',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Is it a group project?',
  'answer': 'No, the capstone is an individual project.\n\nYou can collaborate or discuss a larger idea with other students, but each submitted project must stand on its own. A shared system can work only if it is clearly decomposed into independent projects, where each person has a separate qualifying component and a separate repository.\n\nIf the work cannot be evaluated independently for each participant, it does not satisfy the project requirement.'},
 {'id': '86d99bbf21',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Authentication: Why is my OPENAI_API_KEY not found in the Jupyter notebook?',
  'answer': 'Make sure you installed and used `python-dotenv`.\n\n```bash\npip install python-dotenv\n```\n\nThen load the `.env` file in the notebook before creating the OpenAI client:\n\n```python\nfrom dotenv import load_dotenv\n\nload_doten

In [16]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

0fab61eca2 == 74eb249bbf: False
86d99bbf21 == 74eb249bbf: False
74eb249bbf == 74eb249bbf: True
1a7b27c4df == 74eb249bbf: False
acf8fa5356 == 74eb249bbf: False


In [17]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[0, 0, 1, 0, 0]

In [18]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [19]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

Is it too late to enroll since I just found out about the program?


[0, 0, 1, 0, 0]

In [20]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)

How should I go about submitting my questions during the live streams?


[1, 0, 0, 0, 0]

In [21]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)

Where is the main hub for checking out the upcoming schedule and course milestones?


[0, 0, 1, 0, 0]

In [22]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [23]:
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/590 [00:00<?, ?it/s]

In [24]:
relevance[:15]

[[0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [25]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [26]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [27]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/590 [00:00<?, ?it/s]

In [28]:
sample = relevance_total[:15]

In [29]:
sample

[[0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [51]:
10/15

0.6666666666666666

In [31]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

0.6666666666666666

In [32]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [33]:
hit_rate(relevance)

0.6406779661016949

In [34]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

0.5388888888888889

In [35]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [36]:
mrr(sample)

0.5388888888888889

In [37]:
mrr(relevance)

0.4891242937853109

In [38]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [39]:
evaluate(ground_truth, text_search)

  0%|          | 0/590 [00:00<?, ?it/s]

{'hit_rate': 0.6406779661016949, 'mrr': 0.4891242937853109}

In [40]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [41]:
evaluate(ground_truth, text_search_v2)

  0%|          | 0/590 [00:00<?, ?it/s]

{'hit_rate': 0.6898305084745763, 'mrr': 0.5211016949152539}

In [42]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [43]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/590 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.7677966101694915, 'mrr': 0.6094350282485872}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.7406779661016949, 'mrr': 0.576016949152542}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.6406779661016949, 'mrr': 0.4891242937853109}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.6237288135593221, 'mrr': 0.46542372881355953}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.5983050847457627, 'mrr': 0.4449717514124296}


In [44]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [45]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/590 [00:00<?, ?it/s]

In [46]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.827119,0.663136
19,2.0,4.0,0.2,0.827119,0.663136
35,5.0,10.0,0.5,0.827119,0.663136
7,1.0,4.0,0.2,0.832203,0.663051
6,1.0,4.0,0.1,0.823729,0.661243
8,1.0,4.0,0.5,0.828814,0.660989
4,1.0,2.0,0.2,0.823729,0.659266
18,2.0,4.0,0.1,0.822034,0.658814
34,5.0,10.0,0.2,0.822034,0.657966
33,5.0,10.0,0.1,0.822034,0.657825


## Note: why my hit_rate / MRR differ from the course reference notebook

My final tuned results (hit_rate ≈ 0.827, MRR ≈ 0.663, from the grid search cell above) are
noticeably lower than the reference `02-search-eval.ipynb` (hit_rate ≈ 0.975, MRR ≈ 0.885, same
grid search). I compared both notebooks cell-by-cell, using identical code, identical boost
values, and the identical `evaluate()`/`hit_rate()`/`mrr()` functions, and traced the gap to
differences in the input data rather than a bug in the implementation.

**Side-by-side, same code / same boosts:**

| Stage                                  | Reference (79 docs, 395 Qs) | Mine (118 docs, 590 Qs) |
|-----------------------------------------|:---------------------------:|:------------------------:|
| `text_search` baseline                  | 0.899 / 0.769                | 0.641 / 0.489             |
| `text_search_v2`                        | 0.909 / 0.792                | 0.690 / 0.521             |
| single boost sweep, best value          | 0.924 / 0.814 (boost=1.0)    | 0.768 / 0.609 (boost=0.5) |
| single boost sweep, boost=3.0 (=baseline)| 0.899 / 0.769                | 0.641 / 0.489             |
| single boost sweep, boost=10.0          | 0.858 / 0.712                | 0.598 / 0.445             |
| grid search, best combo                 | 0.975 / 0.885                | 0.827 / 0.663             |

**Root causes identified:**

1. **Bigger index (79 → 118 documents in `documents_llm`, +49%).** `text_search` always
   returns a fixed `num_results=5`. A larger index means more plausible near-matches
   competing for those 5 slots, which mechanically lowers hit_rate/MRR at fixed k —
   independent of code correctness.

2. **Different LLM used for ground-truth question generation.** I used
   `gemini-3.1-flash-lite` instead of the course's model to generate the 5 paraphrased
   questions per FAQ answer. Since `text_search` is pure lexical/TF-IDF matching (no semantic
   understanding), how closely a generated question's wording overlaps with the source FAQ
   text has a direct effect on findability. Different LLMs default to different paraphrasing
   styles, which shifts this overlap.

**Why I'm confident this is a data effect, not an implementation issue:**

The *relative pattern* across every experiment matches the reference almost exactly:
boost=3.0 in the sweep reproduces the plain baseline in both notebooks (sanity check passes),
performance peaks at a low boost value and steadily degrades as boost increases toward 10 in
both notebooks, and full grid search beats every single-parameter attempt in both notebooks.
A real implementation bug would typically produce inconsistent or erratic results, not a
uniformly-shifted-but-identically-shaped curve. The consistent ~15-25 point gap at *every*
single stage points to a harder underlying retrieval problem (more documents, less
lexically-overlapping questions), not a defect in `compute_relevance`, `hit_rate`, or `mrr`.

